# 📊 Exploratory Data Analysis — Fake News Detection

**Step 2** of the FakeShield pipeline.

This notebook explores the merged dataset and produces every plot required by Section 4, Step 2 (2a–2i).


## Setup & Imports

In [ ]:
import os, sys, re, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)

# Make src/ importable so we can reuse the cleaning pipeline.
sys.path.insert(0, os.path.abspath('../src'))
from data_preprocessing import run_preprocessing


## Load & Clean Data
Reuses `data_preprocessing.run_preprocessing()` so the EDA always matches the training data exactly.

In [ ]:
df = run_preprocessing(save=False)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['label_name'] = df['label'].map({1: 'Fake', 0: 'True'})
df.head(3)


## 2a. Class Distribution (Fake vs True)

In [ ]:
ax = sns.countplot(data=df, x='label_name',
                   palette={'Fake': '#d6455d', 'True': '#3ddc84'})
ax.set_title('Class Distribution: Fake vs True')
ax.set_xlabel('Class'); ax.set_ylabel('Article Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom')
plt.show()


## 2b. Subject Distribution by Label (Stacked Bar)

In [ ]:
ct = pd.crosstab(df['subject'], df['label_name'])
ct.plot(kind='bar', stacked=True,
        color={'Fake': '#d6455d', 'True': '#3ddc84'})
plt.title('Subject Distribution by Label')
plt.xlabel('Subject'); plt.ylabel('Count')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()


## 2c. Text Length Distribution (Word Count)

In [ ]:
df['text_wc'] = df['text'].astype(str).str.split().str.len()
plt.hist(df[df.label == 1]['text_wc'], bins=60, alpha=0.6,
         label='Fake', color='#d6455d', range=(0, 2000))
plt.hist(df[df.label == 0]['text_wc'], bins=60, alpha=0.6,
         label='True', color='#3ddc84', range=(0, 2000))
plt.title('Article Body Word-Count Distribution')
plt.xlabel('Word count'); plt.ylabel('Frequency')
plt.legend(); plt.show()


## 2d. Title Length Distribution

In [ ]:
df['title_wc'] = df['title'].astype(str).str.split().str.len()
plt.hist(df[df.label == 1]['title_wc'], bins=40, alpha=0.6,
         label='Fake', color='#d6455d')
plt.hist(df[df.label == 0]['title_wc'], bins=40, alpha=0.6,
         label='True', color='#3ddc84')
plt.title('Title Word-Count Distribution')
plt.xlabel('Title word count'); plt.ylabel('Frequency')
plt.legend(); plt.show()


## 2e. Top 30 Most Frequent Words (Fake vs True)
Stopwords excluded. Shown as both word clouds and bar charts.

In [ ]:
STOP = set('''a an the and or but if while is are was were be been
being to of in on for with as by at from that this these those it
its he she they them his her their we you i me my your our us not no
do does did have has had will would can could should than then so
such into about over after before up down out off again more most
some any all who what when where why how which there here said'''.split())

def top_words(series, n=30):
    tokens = re.findall(r'[a-zA-Z]+',
                        ' '.join(series.astype(str)).lower())
    tokens = [t for t in tokens if t not in STOP and len(t) > 2]
    return Counter(tokens).most_common(n)

fake_top = top_words(df[df.label == 1]['text'])
true_top = top_words(df[df.label == 0]['text'])


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
for a, data, title, cmap in [
    (ax[0], dict(fake_top), 'Fake — Top Words', 'Reds'),
    (ax[1], dict(true_top), 'True — Top Words', 'Greens')]:
    wc = WordCloud(width=600, height=400, background_color='white',
                   colormap=cmap).generate_from_frequencies(data)
    a.imshow(wc); a.axis('off'); a.set_title(title)
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 8))
fw, fc = zip(*fake_top); tw, tc = zip(*true_top)
ax[0].barh(fw[::-1], fc[::-1], color='#d6455d')
ax[0].set_title('Fake — Top 30 Words')
ax[1].barh(tw[::-1], tc[::-1], color='#3ddc84')
ax[1].set_title('True — Top 30 Words')
plt.tight_layout(); plt.show()


## 2f. Publication Date Trend (Monthly, Fake vs True)

In [ ]:
tmp = df.dropna(subset=['date']).copy()
tmp['month'] = tmp['date'].dt.to_period('M').dt.to_timestamp()
trend = tmp.groupby(['month', 'label_name']).size().unstack(fill_value=0)
trend.plot(color={'Fake': '#d6455d', 'True': '#3ddc84'})
plt.title('Monthly Publication Frequency: Fake vs True')
plt.xlabel('Month'); plt.ylabel('Article count')
plt.tight_layout(); plt.show()


## 2g. Average Article Length Over Time

In [ ]:
tmp['len'] = tmp['text'].astype(str).str.split().str.len()
avg_len = tmp.groupby(['month', 'label_name'])['len'].mean()\
             .unstack(fill_value=0)
avg_len.plot(color={'Fake': '#d6455d', 'True': '#3ddc84'})
plt.title('Average Article Word-Count Over Time')
plt.xlabel('Month'); plt.ylabel('Mean word count')
plt.tight_layout(); plt.show()


## 2h. Unique Subjects: Fake vs True

In [ ]:
fake_subj = df[df.label == 1]['subject'].nunique()
true_subj = df[df.label == 0]['subject'].nunique()
print(f'Unique subjects in FAKE: {fake_subj}')
print(f'Unique subjects in TRUE: {true_subj}')
print('\nFake subjects:',
      sorted(df[df.label==1]['subject'].dropna().unique()))
print('True subjects:',
      sorted(df[df.label==0]['subject'].dropna().unique()))


## 2i. Correlation Heatmap of Metadata Numeric Features

In [ ]:
from feature_engineering import engineer_features, METADATA_FEATURE_NAMES
X_meta, y = engineer_features(df, fit=True)
meta_df = pd.DataFrame(X_meta, columns=METADATA_FEATURE_NAMES)
plt.figure(figsize=(13, 10))
sns.heatmap(meta_df.corr(), cmap='coolwarm', center=0,
            annot=False, linewidths=0.3)
plt.title('Correlation Heatmap — Engineered Metadata Features')
plt.tight_layout(); plt.show()


---
### ✅ EDA Complete
Proceed to **Step 3** (feature engineering) and **Step 6** (training): `python src/train.py`.